In [114]:
import pandas as pd 
import requests 
from bs4 import BeautifulSoup
from pathlib import Path
import time 
import random 
import re
AGENTS = [
    'ASTRA', 'BREACH', "BRIMSTONE", "CHAMBER", "CLOVE", 
    'CYPHER', 'DEADLOCK', 'FADE', 'GEKKO','HARBOR', 
    'HUSH', 'ISO', 'JETT', 'KAY/O', 'KILLJOY', 'MIKS', 
    'NEON','PHOENIX','RAZE', 'REYNA', 'SAGE', 'SKYE', 
    'SOVA', 'TEJO', 'VETO','VIPER', 'VYSE', 'WAYLAY', 'YORU'
]
AGENT_SET = set(AGENTS)
NEW_AGENT_RE = re.compile(r'\bnew agent\s+([\w/]+)', re.IGNORECASE)

In [ ]:
r = requests.get("https://playvalorant.com/en-us/news/game-updates/")
with open("data/games_updates.html", "w") as out: 
    out.write(r.text)
soup = BeautifulSoup(r.text)

In [30]:

raw_links = soup.find_all("a")
patch_links = [link for link in raw_links if "Patch" in str(link)]
for i in patch_links[:5]: 
    print(i)
print(len(patch_links))

<a aria-label="VALORANT Patch Notes 13.10" class="sc-eec75e30-0 bMzQwm sc-d9e74f39-0 jHYayn sc-452d1a06-5 bMuTcM action" data-testid="articlefeaturedcard-component" href="/en-us/news/game-updates/valorant-patch-notes-13-10" role="button"><span><div class="sc-6fae0810-0 kZtTQ" data-testid="card"><div class="sc-729df4b-0 linrEJ action" data-testid="card-image"><div class="sc-cf6885cf-0 faINlR" data-aspect-ratio="1.7777777777777777" data-testid="aspect-ratio-container"><div class="innerWrapper"><picture style="display:contents"><source srcset="https://cmsassets.rgpub.io/sanity/images/dsfx7636/news_live/296cff0d10e916f8a529ec9b60da134f0c80a448-1920x1080.png?accountingTag=VAL&amp;fm=webp&amp;q=80&amp;h=350&amp;w=622&amp;fit=crop&amp;crop=center" type="image/webp"/><img alt="" class="sc-3c851706-0 ixavsq sc-f924da1a-0 bkLimp sc-1f4a6b27-0 cBMoRk fit-undefined crop-true" data-testid="mediaImage" loading="lazy" src="https://cmsassets.rgpub.io/sanity/images/dsfx7636/news_live/296cff0d10e916f8a5

In [95]:
links = {link['aria-label']:[link['href'], link.find('time')['datetime']] for link in patch_links }
print(len(links))

100


In [97]:
for key in links: 
    print(f"Name: {key}\n   {links[key][0]}\n   {links[key][1]}")

Name: VALORANT Patch Notes 13.10
   /en-us/news/game-updates/valorant-patch-notes-13-10
   2026-11-10T14:00:00.000Z
Name: VALORANT Patch Notes 13.09
   /en-us/news/game-updates/valorant-patch-notes-1309
   2026-10-27T13:00:00.000Z
Name: VALORANT Patch Notes 13.08
   /en-us/news/game-updates/valorant-patch-notes-13-08
   2026-10-13T13:00:00.000Z
Name: VALORANT Patch Notes 13.06
   /en-us/news/game-updates/valorant-patch-notes-13-06
   2026-09-22T13:00:00.000Z
Name: VALORANT Patch Notes 13.05
   /en-us/news/game-updates/valorant-patch-notes-13-05
   2026-09-01T13:00:00.000Z
Name: VALORANT Patch Notes 13.04
   /en-us/news/game-updates/valorant-patch-notes-13-04
   2026-08-18T13:00:00.000Z
Name: VALORANT Patch Notes 13.02
   /en-us/news/game-updates/valorant-patch-notes-13-02
   2026-07-28T13:00:00.000Z
Name: VALORANT Patch Notes 13.01
   /en-us/news/game-updates/valorant-patch-notes-13-01
   2026-07-14T13:00:00.000Z
Name: VALORANT Patch Notes 13.00
   /en-us/news/game-updates/valorant-pat

In [103]:
r = requests.get("https://playvalorant.com/en-us/news/game-updates/valorant-patch-notes-13-10/")
soup = BeautifulSoup(r.text)

In [ ]:
def _own_strings(li):
    """Text that belongs to this <li> itself, not to any list nested inside it."""
    my_list = li.find_parent(['ul', 'ol'])
    return [s.strip() for s in li.find_all(string=True)
            if s.strip() and s.find_parent(['ul', 'ol']) is my_list]

def _label(li):
    return ' '.join(_own_strings(li))

def _agent_name(li):
    """Return the agent name if this <li> is an agent heading, else None."""
    pieces = _own_strings(li)
    if not pieces:
        return None
    if pieces[0].upper() in AGENT_SET:                   # "Clove", "Jett"
        return pieces[0].upper()
    m = NEW_AGENT_RE.search(' '.join(pieces))            # "New Agent Hush is now in game!"
    if m and m.group(1).upper() in AGENT_SET:
        return m.group(1).upper()
    return None

def _child_lis(li):
    sub = li.find(['ul', 'ol'], recursive=False)
    return sub.find_all('li', recursive=False) if sub else []

def _walk(li, data, date, agent=None, item=pd.NA):
    children = _child_lis(li)
    if agent is None:                       # no agent found yet: check this <li>, else dig deeper
        name = _agent_name(li)
        for child in children:
            _walk(child, data, date, agent=name)
        return
    if children:                            # a heading inside an agent section = the item
        for child in children:
            _walk(child, data, date, agent=agent, item=_label(li))
    else:                                   # a leaf = a description
        data.append({
            'agent': agent,
            'item': item,
            'description': li.get_text(' ', strip=True),
            'buff': pd.NA,
            'date': date,
        })

def add_agent_updates(soup, data, date):
    header = soup.find('h2', string=lambda s: s and 'AGENT' in s.upper() and 'UPDATE' in s.upper())
    if header is None:
        return
    # Read every top-level list until the next <h2>, so an image or <p> in between doesn't cut anything off
    for el in header.find_all_next(['h2', 'ul', 'ol']):
        if el.name == 'h2':
            break
        if el.find_parent('li') is not None:   # nested lists are handled by _walk
            continue
        for li in el.find_all('li', recursive=False):
            _walk(li, data, date)

In [118]:
data = []
base = "https://playvalorant.com/"
for key in links: 
    tab_name = key
    link = base + links[key][0]
    date = links[key][1]
    r = requests.get(link)
    soup = BeautifulSoup(r.text)
    add_agent_updates(soup, data, date)
    print("tab_name:", tab_name)
    time.sleep(random.uniform(1.5, 3))

df = pd.DataFrame(data)
df.head()

tab_name: VALORANT Patch Notes 13.10
tab_name: VALORANT Patch Notes 13.09
tab_name: VALORANT Patch Notes 13.08
tab_name: VALORANT Patch Notes 13.06
tab_name: VALORANT Patch Notes 13.05
tab_name: VALORANT Patch Notes 13.04
tab_name: VALORANT Patch Notes 13.02
tab_name: VALORANT Patch Notes 13.01
tab_name: VALORANT Patch Notes 13.00
tab_name: VALORANT Patch Notes 12.11
tab_name: VALORANT Patch Notes 12.10
tab_name: VALORANT Patch Notes 12.09
tab_name: VALORANT Patch Notes 12.08
tab_name: VALORANT Patch Notes 12.07
tab_name: VALORANT Patch Notes 12.06
tab_name: VALORANT Patch Notes 12.05
tab_name: VALORANT Patch Notes 12.04
tab_name: VALORANT Patch Notes 12.03
tab_name: VALORANT Patch Notes 12.02
tab_name: VALORANT Patch Notes 12.01
tab_name: VALORANT Patch Notes 12.00
tab_name: VALORANT Patch Notes 11.11
tab_name: VALORANT Patch Notes 11.10
tab_name: VALORANT Patch Notes 11.09
tab_name: VALORANT Patch Notes 11.08
tab_name: Patch 11.08 Gameplay Updates
tab_name: Patch 11.08 Gameplay Updat

,agent,item,description,buff,date
0,CYPHER,Spycam,Cooldown after being destroyed decreased from ...,<NA>,2026-11-10T14:00:00.000Z
1,CYPHER,Spycam,"Reclaim cooldown increased from 15s >>> 20s, i...",<NA>,2026-11-10T14:00:00.000Z
2,KILLJOY,Turret,Cooldown after being destroyed decreased from ...,<NA>,2026-11-10T14:00:00.000Z
3,VYSE,Arc Rose,Cooldown after being destroyed decreased from ...,<NA>,2026-11-10T14:00:00.000Z
4,SAGE,NaN,Increased accuracy while targeting allies,<NA>,2026-11-10T14:00:00.000Z


In [119]:
df.to_csv("data/patch_notes.csv", index=False)